# Data Loading

In [ ]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [ ]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
#reviews_df.head()

In [ ]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

In [ ]:
narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

In [ ]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


In [ ]:
# Cleaning up narrative games by columns that are relevant to us
narrative_games_cleaned = narrative_games[['AppID', 'Name', 'Release date', 'Required age', 'About the game', 'Categories', 'Genres', 'Tags']]

available_names = filtered_reviews["app_name"].unique()
narrative_games_available = narrative_games_cleaned[narrative_games_cleaned["AppID"].isin(available_names)]

review_count = []
pos_ratio = []
pos_count = []
neg_count = []
for i in range (len(narrative_games_available)):
    game = narrative_games_available.iloc[i]
    game_reviews = filtered_reviews[filtered_reviews['app_name']==game.get('AppID')]
    game_review_count = len(game_reviews)
    game_postiive_reviews = sum(1 for review in game_reviews['review_score'] if review >0)
    game_pos_ratio = game_postiive_reviews/game_review_count
    game_negative_reviews = game_review_count-game_postiive_reviews
    review_count.append(game_review_count)
    pos_ratio.append(game_pos_ratio)
    pos_count.append(game_postiive_reviews)
    neg_count.append(game_negative_reviews)
    
narrative_games_available['positive'] = pos_count
narrative_games_available['negative'] = neg_count
narrative_games_available['review_count'] = review_count
narrative_games_available['pos_ratio'] = pos_ratio

narrative_games_available.head()

## Data Cleaning

In [ ]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews.head()

In [ ]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews.head()

In [ ]:
# Getting highest rating based on number of reviews and number of positive reviews
top5_df = narrative_games_available.nlargest(6, ["review_count", "pos_ratio"], 'first')
top5_games = []

for i in range (len(top5_df)):
    game = top5_df.iloc[i]
    top5_games.append(game.get("AppID")) #Get all the game names
    
#weird thing of data where Portal 2 appears twice
top5_df = top5_df.drop(top5_df.index[1])
top5_games.pop(1)

top5_df

In [ ]:
# Did it this way since games with no reviews will register as 0 and will clog up the space for lowest rated games
# We still want a significant amount of reviews to base what is actually bad vs what is underrepresented/underreviewed
cond = narrative_games_available.get("review_count") >= 1000
temp_bot_df = narrative_games_available.where(cond)
bot5_df = temp_bot_df.nsmallest(5, "pos_ratio", 'first')

bot5_games = []

for i in range (len(bot5_df)):
    game = bot5_df.iloc[i]
    bot5_games.append(game.get("AppID")) #Get all the game names

bot5_df.head()
#print(bot5_games)

# Exploratory Data Analysis

In [ ]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices

estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
filtered_reviews.head()

In [ ]:
all_top5_df = filtered_reviews[filtered_reviews['app_name'].isin(top5_games)]
all_bot5_df = filtered_reviews[filtered_reviews['app_name'].isin(bot5_games)]


In [ ]:
all_topbot_reviews = pd.concat([all_top5_df,all_bot5_df])
all_topbot_reviews.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
from wordcloud import WordCloud

In [ ]:
def create_wordcloud(reviews, max_words=500):
    """Create a wordcloud of most common words in a set of tweets"""
    
    # Transform text for WordCloud
    reviews = reviews.dropna()
    reviews = ' '.join(reviews)
    reviews = reviews.replace(' ', ',')
    
    # Generate wordcloud image
    wc = WordCloud(background_color="white", max_words=max_words, colormap='plasma')
    wc.generate(reviews)
    plt.imshow(wc, interpolation='bilinear')
    plt.title('Review Cloud', size=30)
    plt.axis("off")

# Generate a wordcloud
create_wordcloud(filtered_reviews['review_text'])

### enable gpu

In [ ]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# Sentiment Analysis

### Without Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews, batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()
filtered_reviews.to_csv("output_senti.csv")

### Top 5/Bottom 5 Data

In [ ]:

review_docs = nlp_with_sentiment.pipe(all_topbot_reviews["lemmatized_reviews"], batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(all_topbot_reviews)))

all_topbot_reviews['com_sent'] = com_sent
all_topbot_reviews['pos_sent'] = pos_sent
all_topbot_reviews['neg_sent'] = neg_sent
all_topbot_reviews['neu_sent'] = neu_sent

all_topbot_reviews.head()
all_topbot_reviews.to_csv("output_senti_all.csv")

In [ ]:
print(sum(1 for review in all_topbot_reviews['com_sent'] if review < -0.90))
print(sum(1 for review in all_topbot_reviews['review_score'] if review < 0))
print(sum(1 for review in all_topbot_reviews['com_sent'] if review > 0.90))
print(sum(1 for review in all_topbot_reviews['review_score'] if review > 0 ))

In [ ]:
#run if you skipped the senti analysis
all_topbot_reviews = pd.read_csv('output_senti_all.csv', index_col=False)

all_topbot_reviews.head()

In [ ]:
all_top5_df=all_topbot_reviews[all_topbot_reviews["app_name"].isin(top5_games)]

print(sum(1 for review in all_top5_df['com_sent'] if review > 0.90))
print(sum(1 for review in all_top5_df['review_score'] if review > 0 ))

all_top5_df.head()

In [ ]:
all_bot5_df=all_topbot_reviews[all_topbot_reviews["app_name"].isin(bot5_games)]

print(sum(1 for review in all_topbot_reviews['com_sent'] if review < 0))
print(sum(1 for review in all_topbot_reviews['review_score'] if review < 0))

all_bot5_df.head()

In [ ]:
#getting the 10000 most positive and most negative sentiments

extreme_top5_df = all_top5_df.nlargest(10000, ["com_sent"], 'first')
extreme_bot5_df = all_bot5_df.nsmallest(10000, ["com_sent"], 'first')

# Topic Modeling

References:

* https://maartengr.github.io/BERTopic/getting_started/quickstart/quickstart.html
* https://maartengr.github.io/BERTopic/getting_started/visualization/visualization.html
* https://maartengr.github.io/BERTopic/getting_started/tips_and_tricks/tips_and_tricks.html

We want to perform topic modeling on:
* positive reviews from the top 5 games
* negative reviews from the bottom 5 games
* all revies from each game in the top 5 and bottom 5

NOTE: Not recommended daw to manually stopword removal and lemmatization for BERTopic modeling. There's some tips for removing/reducing stopwords in the third ref above - moi

In [ ]:
TM_SAMPLE_N = 10000

### CRITERIA:
### - sentence form
### - sentence has at least 5 words
### - no need for stopword removal, no need for lematization

def get_review_sentences(reviews_list, min_word_count=5):
    sentences = []
    for i, review in enumerate(reviews_list):
        for sentence in (str(review)).split(" | "):
            if (len(sentence.split()) >= min_word_count):
                sentences.append(("").join(sentence))

    print("Sentences retrieved: " + str(len(sentences)))
    return sentences

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import TfidfVectorizer


TM_DEFAULT_MIN_TOPIC_SIZE = 200
RANDOM_SEED = 230995 
TM_OUTPUT_FOLDER = "TopicModelingOutputs"

sentence_model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
vectorizer_model = CountVectorizer(stop_words="english")

In [ ]:
vectorizer_model = CountVectorizer(stop_words="english")

In [ ]:
umap_model = UMAP(n_neighbors=15, 
                  n_components=5, 
                  min_dist=0.0, 
                  metric='cosine'
            )

In [ ]:
def get_filesafe_name(original_name: str):
    alphanumeric_only = re.sub(r'[^a-zA-Z0-9 ]', '', original_name)
    result = re.sub(r' +', '_', alphanumeric_only)
    return result

In [ ]:
def perform_topic_modeling(data, min_cluster_size=TM_DEFAULT_MIN_TOPIC_SIZE, data_title="", output_subfolder="", special_vectorizer=False, custom_nr_topics=20):
    MIN_TOPIC_COUNT_WANTED = 5
    
    hdbscan_model = HDBSCAN(min_cluster_size=min_cluster_size, 
                        metric='euclidean',
                        cluster_selection_method='eom', 
                        prediction_data=False,
                    )

    vectorizer_to_use = vectorizer_model
    if (special_vectorizer!=False):
        vectorizer_to_use = special_vectorizer
    
    topic_model = BERTopic(hdbscan_model=hdbscan_model, 
                           vectorizer_model=vectorizer_to_use, 
                           umap_model=umap_model,
                           calculate_probabilities=False,
                           nr_topics=custom_nr_topics,
                           )
    
    embeddings = sentence_model.encode(data, show_progress_bar=True, batch_size=32, device="cuda")
    print("embeddings len: " + str(len(embeddings)))
    topic_model.fit(data, embeddings=embeddings)

    ## IT WILL ERROR IF THERE ARENT ENOUGH TOPICS
    ## so make sure to check
    topic_info = topic_model.get_topic_info()
    topics_count = len(set(topic_info["Topic"])) - 1 # - 1 since there's also a category for outlier topics
    while topics_count < MIN_TOPIC_COUNT_WANTED:
        print(f"Only {topics_count} topics found")
        print("Not enough topics found when min_cluster_size=" + str(hdbscan_model.min_cluster_size) + ". Reducing to 2/3 and then rerunning.")
        topic_model.hdbscan_model.set_params(min_cluster_size=int(hdbscan_model.min_cluster_size*(2/3)))
        topic_model.fit(data)
        topic_info = topic_model.get_topic_info()
        topics_count = len(set(topic_info["Topic"])) - 1

    graph_label = data_title + " | " if (data_title != "") else ""
    vistopics = topic_model.visualize_topics(title=(graph_label + "Intertopic Distance Map"), width=1200, height=800)
    visdocus = topic_model.visualize_documents(data, embeddings=embeddings, title=(graph_label + "Documents and Topics"), width=1200, height=800)

    output_filename = data_title if (data_title != "") else data[0][:64] ## use first line of data if no data title provided
    output_subfolder = (output_subfolder + "/") if output_subfolder != "" else ""

    vistopics.write_image(f'{TM_OUTPUT_FOLDER}/{output_subfolder}{get_filesafe_name(output_filename)}-topics.png')
    visdocus.write_image(f'{TM_OUTPUT_FOLDER}/{output_subfolder}{get_filesafe_name(output_filename)}-documents.png')    
    vistopics.write_html(f'{TM_OUTPUT_FOLDER}/{output_subfolder}{get_filesafe_name(output_filename)}-topics.html')
    visdocus.write_html(f'{TM_OUTPUT_FOLDER}/{output_subfolder}{get_filesafe_name(output_filename)}-documents.html')

    return topic_info, vistopics, visdocus


### Top 10,000 most Positive Reviews from Top 5 Games

In [ ]:
from sklearn.feature_extraction import text

In [ ]:
gamename_stopwords = ["portal", "bioshock", "bio shock", "undertale", "fallout", "witcher",
                      "batman", "batmans", "starwars", "star wars", "elder scrolls", "skyrim", "mafia"]

specialvec_stopwords = list(text.ENGLISH_STOP_WORDS.union(gamename_stopwords))
vectorizer_10k = CountVectorizer(stop_words=specialvec_stopwords)

## REFS:
# https://towardsdatascience.com/basics-of-countvectorizer-e26677900f9c/
# https://stackoverflow.com/questions/24386489/adding-words-to-scikit-learns-countvectorizers-stop-list 

In [ ]:
print(len(extreme_top5_df))
extreme_top5_reviews = (extreme_top5_df["review_text"].to_list())
extreme_top5_sentences = get_review_sentences(extreme_top5_reviews)

In [ ]:
top5_topicinfo, top5_topicvis, top5_docusvis = perform_topic_modeling(extreme_top5_sentences, 
                                                                      min_cluster_size=400,
                                                                      data_title="Positive Reviews from Top 5 Games",
                                                                      special_vectorizer=vectorizer_10k
                                                                      )
top5_topicvis.show()
top5_docusvis.show()

### All Negative Reviews from Bottom 5 Games

In [ ]:
print(len(extreme_bot5_df["review_text"].to_list()))
extreme_bot5_reviews = extreme_bot5_df["review_text"].to_list()
extreme_bot5_sentences = get_review_sentences(extreme_bot5_reviews)

In [ ]:
all_bot5_topicinfo, all_bot5_topicvis, all_bot5_docusvis = perform_topic_modeling(extreme_bot5_sentences, 
                                                                                  min_cluster_size = 250,
                                                                                  data_title="Negative Reviews from Bottom 5 Games",
                                                                                  special_vectorizer=vectorizer_10k,
                                                                                  custom_nr_topics=25)
all_bot5_topicvis.show()
all_bot5_docusvis.show()

In [ ]:
all_top5_topicinfo0.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/all_top5.csv', index=False)
all_bot5_topicinfo0.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/all_bot5.csv', index=False)

### Topic Modeling Per Game

In [ ]:
## BUG: Portal 2 appears in 'top5_games twice
# temp workaround is this cell

top5_games = list(set(top5_games))
print(top5_games)
bot5_games = list(set(bot5_games))
print(bot5_games)

In [ ]:
def perform_per_t5b5_topicmodeling(is_top5: bool, list_index: int, custom_mcs: int=-1, show_graphs=True):
    if (is_top5):
        df = all_top5_df
        game_name = str(top5_games[list_index])
        output_subfolder = "top5"
    else:
        df = all_bot5_df
        game_name = str(bot5_games[list_index])
        output_subfolder = "bot5"

    reviews = (df[df.app_name==game_name])["review_text"]
    print(game_name)
    sentences = get_review_sentences(reviews)

    if (custom_mcs==-1):
        if ((len(sentences)) > 5000):
            mcs = int(len(sentences) * 0.01)
        else:
            mcs = int(len(sentences) * 0.05)
        print("Using calculated mcs: " + str(mcs))
    else:
        mcs = custom_mcs
        print("Using custom mcs: " + str(mcs))

    topic_info, topic_vis, docus_vis = perform_topic_modeling(sentences, 
                                                            min_cluster_size=mcs, 
                                                            data_title=game_name,
                                                            output_subfolder=output_subfolder,
                                                            )

    if(show_graphs):
        topic_vis.show()
        docus_vis.show()

    return topic_info, topic_vis, docus_vis

In [ ]:
t1_topicinfo, t1_topicvis, t1_docusvis = perform_per_t5b5_topicmodeling(True, 0, custom_mcs=400)

In [ ]:
t2_topicinfo, t2_topicvis, t2_docusvis = perform_per_t5b5_topicmodeling(True, 1, custom_mcs=500)

In [ ]:
t3_topicinfo, t3_topicvis, t3_docusvis = perform_per_t5b5_topicmodeling(True, 2, custom_mcs=600)

In [ ]:
t4_topicinfo, t4_topicvis, t4_docusvis = perform_per_t5b5_topicmodeling(True, 3, custom_mcs=500)

In [ ]:
t5_topicinfo, t5_topicvis, t5_docusvis = perform_per_t5b5_topicmodeling(True, 4, custom_mcs=650)

In [ ]:
b1_topicinfo, b1_topicvis, b1_docusvis = perform_per_t5b5_topicmodeling(False, 0)

In [ ]:
b2_topicinfo, b2_topicvis, b2_docusvis = perform_per_t5b5_topicmodeling(False, 1)

In [ ]:
b3_topicinfo, b3_topicvis, b3_docusvis = perform_per_t5b5_topicmodeling(False, 2, custom_mcs=40)

In [ ]:
b4_topicinfo, b4_topicvis, b4_docusvis = perform_per_t5b5_topicmodeling(False, 3, custom_mcs=60)

In [ ]:
b5_topicinfo, b5_topicvis, b5_docusvis = perform_per_t5b5_topicmodeling(False, 4, custom_mcs=60)

In [ ]:
t1_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(top5_games[0]))}.csv', index=False)
t2_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(top5_games[1]))}.csv', index=False)
t3_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(top5_games[2]))}.csv', index=False)
t4_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(top5_games[3]))}.csv', index=False)
t5_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(top5_games[4]))}.csv', index=False)

In [ ]:
b1_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(bot5_games[0]))}.csv', index=False)
b2_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(bot5_games[1]))}.csv', index=False)
b3_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(bot5_games[2]))}.csv', index=False)
b4_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(bot5_games[3]))}.csv', index=False)
b5_topicinfo.to_csv(f'{TM_OUTPUT_FOLDER}/topic-info/{(get_filesafe_name(bot5_games[4]))}.csv', index=False)